In [1]:
# ================================================================
# AGCC INTERSTATE ROW WIDTH WORKFLOW
# 1) Load ROW polygons, dissolved interstate centerlines, 2000-ft buffer
# 2) Reuse or generate 1-mile transects (spacing 5280 ft, half-length below)
# 3) Console-only diagnostic of 1-mile transects vs buffer
# 4) Split buffer with 1-mile transects -> Segment_ID, export immediately
# 5) Split ROW with 1-mile transects -> singlepart -> ROW_ID -> Segment_ID
# 6) Reuse or generate 100-ft transects, clip to ROW, median width per ROW
# 7) Export clipped transects and final ROW-width polygons
# NOTE: centerlines are NOT exploded (linemerge only; see generate_transects)
# ================================================================
import os, time, logging
import numpy as np, pandas as pd, geopandas as gpd
from datetime import timedelta
from shapely.ops import unary_union, polygonize, linemerge, split
from shapely.geometry import LineString
from shapely.strtree import STRtree
from shapely.validation import make_valid

# ---------------- SETTINGS ----------------
ROOT_DIR = r"C:\Users\KyleSteen.AzureAD\Documents\ROW_Width_workspace"
J = lambda f: os.path.join(ROOT_DIR, f)
ROW_GPKG               = J("AGCC_Final_ROW_CONUS_10_1.gpkg")
INTERSTATE_LINE_GPKG   = J("Interstates_CONUS_Final_10_1.gpkg")
INTERSTATE_BUFFER_GPKG = J("Interstates_CONUS_Final_10_1_Buffer2000ft.gpkg")
MILE_TRANSECTS_GPKG    = J("INTERSTATE_1_MILE_TRANSECTS.gpkg")
TRANSECTS_100FT_GPKG   = J("INTERSTATE_100FT_TRANSECTS.gpkg")
OUTPUT_GPKG            = J("AGCC_Final_ROWs_CONUS_ROW_Width.gpkg")
SEGMENTS_GPKG          = J("INTERSTATE_BUFFER_SEGMENTS.gpkg")
CLIPPED_TRANSECTS_100FT_GPKG = J("ROW_CLIPPED_100FT_TRANSECTS.gpkg")

TARGET_CRS = "EPSG:5070"
FT = 0.3048
MIN_PIECE_AREA_M2 = 0.01
LOG_EVERY_ROWS = 1000
DIAGNOSTIC_LOG_EVERY = 1000
GEOM_TOL_M = 0.01
WIDTH_IN_FEET = True

MILE_TRANSECT_SPACING_FT = 5280
MILE_TRANSECT_HALF_LENGTH_FT = 4000
MILE_TRANSECT_LENGTH_FT = MILE_TRANSECT_HALF_LENGTH_FT * 2
MILE_TRANSECT_SPACING_M = MILE_TRANSECT_SPACING_FT * FT
MILE_TRANSECT_HALF_LENGTH_M = MILE_TRANSECT_HALF_LENGTH_FT * FT
MILE_TRANSECT_LENGTH_M = MILE_TRANSECT_LENGTH_FT * FT

HUNDRED_FT_TRANSECT_SPACING_FT = 100
HUNDRED_FT_TRANSECT_HALF_LENGTH_FT = 2000
HUNDRED_FT_TRANSECT_LENGTH_FT = HUNDRED_FT_TRANSECT_HALF_LENGTH_FT * 2
HUNDRED_FT_TRANSECT_SPACING_M = HUNDRED_FT_TRANSECT_SPACING_FT * FT
HUNDRED_FT_TRANSECT_HALF_LENGTH_M = HUNDRED_FT_TRANSECT_HALF_LENGTH_FT * FT
HUNDRED_FT_TRANSECT_LENGTH_M = HUNDRED_FT_TRANSECT_LENGTH_FT * FT

# ---------------- LOGGING ----------------
logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger()
script_start = time.time()
log = logger.info

def hms(s): return str(timedelta(seconds=int(max(s, 0))))

def eta_str(done, total, t_start):
    if done <= 0: return "calculating..."
    rate = done / max(time.time() - t_start, 1e-9)
    return hms(max(total - done, 0) / rate)

def progress(i, total, t0, every, msg):
    if i == 1 or i % every == 0 or i == total:
        log(f"  {i:,}/{total:,} ({i / max(total, 1):.1%}) | {msg} | ETA {eta_str(i, total, t0)}")

def banner(title):
    log("-" * 70); log(title); log("-" * 70)

# ---------------- HELPERS ----------------
def remove_existing(path):
    if os.path.exists(path):
        log(f"  Removing existing output:\n    {path}")
        os.remove(path)

def clean_geometry(geom):
    if geom is None or geom.is_empty: return None
    try:
        if not geom.is_valid: geom = make_valid(geom)
    except Exception:
        try: geom = geom.buffer(0)
        except Exception: return None
    return None if geom is None or geom.is_empty else geom

def _extract(geom, kinds):
    if geom is None or geom.is_empty: return []
    if geom.geom_type in kinds[0]: return [geom]
    if geom.geom_type in kinds[1]: return list(geom.geoms)
    if geom.geom_type == "GeometryCollection":
        return [g for p in geom.geoms for g in _extract(p, kinds)]
    return []

def extract_lines(geom):    return _extract(geom, (("LineString",), ("MultiLineString",)))
def extract_polygons(geom): return _extract(geom, (("Polygon",), ("MultiPolygon",)))

def make_transect(line, distance, half_length_m):
    """Perpendicular transect centered on a LineString at `distance`."""
    try:
        pt = line.interpolate(distance)
        delta = min(1.0, max(line.length * 0.001, 0.01))
        p1 = line.interpolate(max(0, distance - delta))
        p2 = line.interpolate(min(line.length, distance + delta))
        dx, dy = p2.x - p1.x, p2.y - p1.y
        mag = (dx * dx + dy * dy) ** 0.5
        if mag == 0: return None
        nx, ny = -dy / mag, dx / mag
        return LineString([(pt.x + nx * half_length_m, pt.y + ny * half_length_m),
                           (pt.x - nx * half_length_m, pt.y - ny * half_length_m)])
    except Exception:
        return None

def generate_transects(lines_gdf, spacing_m, half_length_m, transect_type):
    """Transects along centerlines. No explode: each feature is linemerged, and
    only genuinely disconnected pieces are measured separately (interpolate()
    requires a single LineString)."""
    log(f"  Generating {transect_type} transects...")
    out, total, t0 = [], len(lines_gdf), time.time()
    for i, geom in enumerate(lines_gdf.geometry, start=1):
        if geom is not None and not geom.is_empty:
            g = linemerge(geom) if geom.geom_type == "MultiLineString" else geom
            for part in ([g] if g.geom_type == "LineString" else extract_lines(g)):
                if part.length <= 0: continue
                d = 0.0
                while d <= part.length:
                    t = make_transect(part, d, half_length_m)
                    if t is not None: out.append(t)
                    d += spacing_m
        progress(i, total, t0, 100, f"Transects: {len(out):,}")
    log(f"  Generated transects: {len(out):,}")
    return gpd.GeoDataFrame({"Transect_ID": np.arange(1, len(out) + 1, dtype=np.int64)},
                            geometry=out, crs=TARGET_CRS)

def validate_existing_transects(gdf, expected_length_m, transect_type):
    if gdf is None or len(gdf) == 0: return False
    ok = [g for g in gdf.geometry if g is not None and not g.is_empty
          and g.geom_type in ("LineString", "MultiLineString")]
    if not ok: return False
    log(f"  Existing {transect_type} transects contain {len(ok):,} usable line geometries.")
    log(f"  Existing {transect_type} median length: {float(np.median([g.length for g in ok])) / FT:,.1f} ft")
    log(f"  Expected {transect_type} length: {expected_length_m / FT:,.1f} ft")
    return True

def load_gpkg(path, label):
    log(f"  Reading {label}:\n    {path}")
    gdf = gpd.read_file(path)
    log(f"    Features: {len(gdf):,}\n    CRS: {gdf.crs}")
    if gdf.crs is None: raise ValueError(f"{label} has no CRS.")
    if gdf.crs.to_string() != TARGET_CRS:
        log(f"    Reprojecting to {TARGET_CRS}")
        gdf = gdf.to_crs(TARGET_CRS)
    log(f"    Final CRS: {gdf.crs}")
    return gdf

def get_transects(path, label, spacing_m, half_m, length_m, length_ft, half_ft, spacing_ft):
    """Reuse existing transect file if usable, otherwise generate and export."""
    existing = None
    if os.path.exists(path):
        log(f"  Existing {label} transect file found:\n    {path}")
        existing = gpd.read_file(path)
        log(f"    Features: {len(existing):,}\n    CRS: {existing.crs}")
        if existing.crs is not None and existing.crs.to_string() != TARGET_CRS:
            log(f"    Reprojecting to {TARGET_CRS}")
            existing = existing.to_crs(TARGET_CRS)
    else:
        log(f"  No existing {label} transect file found.")
    if existing is not None and validate_existing_transects(existing, length_m, label):
        log(f"  Reusing existing {label} transects.")
        return existing.copy()
    log(f"  Generating new {label} transects.")
    log(f"  Spacing: {spacing_ft:,} ft\n  Total length: {length_ft:,} ft\n  Half-length: {half_ft:,} ft")
    t = generate_transects(line_gdf, spacing_m, half_m, label)
    log(f"  Exporting generated {label} transects:\n    {path}")
    remove_existing(path)
    t.to_file(path, driver="GPKG")
    return t

# ---------------- SPLITTING ----------------
def split_polygon_with_transects(polygon, cut_lines):
    """Polygonize boundary + transects; fall back to sequential split()."""
    polygon = clean_geometry(polygon)
    if polygon is None: return []
    if not cut_lines: return extract_polygons(polygon)
    try:
        pieces = []
        for piece in polygonize(unary_union([polygon.boundary] + cut_lines)):
            if piece.is_empty or piece.area < MIN_PIECE_AREA_M2: continue
            rp = piece.representative_point()
            try: inside = polygon.covers(rp)
            except Exception: inside = polygon.buffer(0).covers(rp)
            if inside: pieces.append(piece)
        if pieces: return pieces
    except Exception:
        pass
    pieces = [polygon]
    for cut in cut_lines:
        if cut is None or cut.is_empty: continue
        nxt = []
        for p in pieces:
            try:
                if not p.intersects(cut): nxt.append(p); continue
                ex = extract_polygons(split(p, cut))
                nxt.extend(ex if ex else [p])
            except Exception:
                nxt.append(p)
        pieces = nxt
    return [p for p in pieces if p is not None and not p.is_empty and p.area >= MIN_PIECE_AREA_M2]

def split_polys(gdf, transects_gdf, label, out_label):
    """Split every polygon in gdf with the transects (used for buffer and ROW)."""
    log("  Building transect spatial index...")
    tgeoms = list(transects_gdf.geometry)
    tree = STRtree(tgeoms)
    log(f"  Transects indexed: {len(tgeoms):,}\n  Splitting {label}...")
    records, total, t0 = [], len(gdf), time.time()
    for i, (_, row) in enumerate(gdf.iterrows(), start=1):
        polygon = clean_geometry(row.geometry)
        if polygon is not None:
            cuts = [tgeoms[int(k)] for k in tree.query(polygon)
                    if tgeoms[int(k)] is not None and not tgeoms[int(k)].is_empty
                    and polygon.intersects(tgeoms[int(k)])]
            base = row.drop(labels=["geometry"]).to_dict()
            for piece in split_polygon_with_transects(polygon, cuts):
                records.append({**base, "geometry": piece})
        progress(i, total, t0, LOG_EVERY_ROWS, f"{out_label}: {len(records):,}")
    result = gpd.GeoDataFrame(records, crs=gdf.crs)
    log(f"  {out_label}: {len(result):,}")
    return result

# ---------------- 100-FT TRANSECT CLIPPING ----------------
def clip_transects_to_rows(transects_gdf, rows_gdf):
    log("  Building ROW spatial index...")
    rgeoms = list(rows_gdf.geometry)
    row_ids = rows_gdf["ROW_ID"].to_numpy()
    tree = STRtree(rgeoms)
    log("  Clipping 100-ft transects to ROW...")
    records, total, t0 = [], len(transects_gdf), time.time()
    for i, (_, row) in enumerate(transects_gdf.iterrows(), start=1):
        transect = row.geometry
        if transect is not None and not transect.is_empty:
            base = row.drop(labels=["geometry"]).to_dict()
            for k in tree.query(transect):
                rg = rgeoms[int(k)]
                if rg is None or rg.is_empty or not transect.intersects(rg): continue
                try: clipped = transect.intersection(rg)
                except Exception:
                    try: clipped = transect.buffer(0).intersection(rg)
                    except Exception: continue
                for ln in extract_lines(clipped):
                    if ln is None or ln.is_empty or ln.length <= GEOM_TOL_M: continue
                    records.append({**base, "ROW_ID": row_ids[int(k)], "geometry": ln})
        progress(i, total, t0, LOG_EVERY_ROWS, f"Clipped transects: {len(records):,}")
    result = gpd.GeoDataFrame(records, crs=transects_gdf.crs)
    log(f"  Clipped transects created: {len(result):,}")
    return result

# ================================================================
# MAIN WORKFLOW
# ================================================================
log("=" * 70); log("AGCC INTERSTATE ROW WIDTH WORKFLOW"); log("=" * 70)
log(f"Target CRS: {TARGET_CRS}")
log(f"1-mile transect spacing: {MILE_TRANSECT_SPACING_FT:,} ft | length: {MILE_TRANSECT_LENGTH_FT:,} ft | half-length: {MILE_TRANSECT_HALF_LENGTH_FT:,} ft")
log(f"100-ft transect spacing: {HUNDRED_FT_TRANSECT_SPACING_FT:,} ft | length: {HUNDRED_FT_TRANSECT_LENGTH_FT:,} ft | half-length: {HUNDRED_FT_TRANSECT_HALF_LENGTH_FT:,} ft")
log(f"ROW input: {ROW_GPKG}\nInterstate centerlines: {INTERSTATE_LINE_GPKG}\nInterstate buffer: {INTERSTATE_BUFFER_GPKG}")
log("=" * 70)

# ---- STEP 1 ----
banner("STEP 1: LOAD INPUT DATA")
row_gdf    = load_gpkg(ROW_GPKG, "ROW polygons")
line_gdf   = load_gpkg(INTERSTATE_LINE_GPKG, "interstate centerlines")   # not exploded
buffer_gdf = load_gpkg(INTERSTATE_BUFFER_GPKG, "interstate buffer")
log("  Input data loaded successfully.")

# ---- STEP 2 ----
banner("STEP 2: PREPARE 1-MILE TRANSECTS")
mile_transects = get_transects(MILE_TRANSECTS_GPKG, "1-mile", MILE_TRANSECT_SPACING_M,
                               MILE_TRANSECT_HALF_LENGTH_M, MILE_TRANSECT_LENGTH_M,
                               MILE_TRANSECT_LENGTH_FT, MILE_TRANSECT_HALF_LENGTH_FT,
                               MILE_TRANSECT_SPACING_FT)
log(f"  Final 1-mile transects: {len(mile_transects):,}")

# ---- STEP 3 ----
banner("STEP 3: 1-MILE TRANSECT BUFFER DIAGNOSTIC")
log("  Diagnostic is console-only.\n  No diagnostic GeoPackage will be exported.")
buffer_union = unary_union([g for g in buffer_gdf.geometry if g is not None and not g.is_empty])
total_t = len(mile_transects)
crossing = no_inter = suspicious = 0
suspicious_ids = []
t0 = time.time()
log(f"  Total transects: {total_t:,}")
for i, (_, row) in enumerate(mile_transects.iterrows(), start=1):
    tr = row.geometry
    bad = tr is None or tr.is_empty
    if not bad:
        try:
            if tr.intersects(buffer_union): crossing += 1
            else: no_inter += 1
        except Exception:
            bad = True
    if bad:
        suspicious += 1
        if len(suspicious_ids) < 50: suspicious_ids.append(row.get("Transect_ID", i))
    progress(i, total_t, t0, DIAGNOSTIC_LOG_EVERY,
             f"Crossing buffer: {crossing:,} | No intersection: {no_inter:,} | Suspicious/errors: {suspicious:,}")
log("  Diagnostic complete.")
log(f"    Total transects: {total_t:,}\n    Crossing buffer: {crossing:,}\n    No intersection: {no_inter:,}\n    Suspicious/errors: {suspicious:,}")
if suspicious_ids: log(f"    First suspicious transect IDs:\n      {suspicious_ids}")

# ---- STEP 4 ----
banner("STEP 4: SPLIT INTERSTATE BUFFER")
buffer_segments = split_polys(buffer_gdf, mile_transects, "interstate buffer polygons", "Segments created")
buffer_segments = buffer_segments[buffer_segments.geometry.notna() & ~buffer_segments.geometry.is_empty].reset_index(drop=True)
buffer_segments["Segment_ID"] = np.arange(1, len(buffer_segments) + 1, dtype=np.int64)
log(f"  Final buffer segments: {len(buffer_segments):,}")

banner("EXPORTING BUFFER SEGMENTS")
log(f"  Exporting immediately after creation:\n    {SEGMENTS_GPKG}")
remove_existing(SEGMENTS_GPKG)
buffer_segments.to_file(SEGMENTS_GPKG, driver="GPKG")
log("  Buffer segment export complete.")

# ---- STEP 5 ----
banner("STEP 5: SPLIT ROW USING 1-MILE TRANSECTS")
split_row_gdf = split_polys(row_gdf, mile_transects, "ROW polygons", "ROW pieces created")
log(f"  ROW pieces before singlepart: {len(split_row_gdf):,}")

# ---- STEP 6 ----
banner("STEP 6: MULTIPART TO SINGLEPART + ROW_ID")
sp_records, total, t0 = [], len(split_row_gdf), time.time()
for i, (_, row) in enumerate(split_row_gdf.iterrows(), start=1):
    base = row.drop(labels=["geometry"]).to_dict()
    for poly in extract_polygons(row.geometry):
        if poly is not None and not poly.is_empty and poly.area >= MIN_PIECE_AREA_M2:
            sp_records.append({**base, "geometry": poly})
    progress(i, total, t0, LOG_EVERY_ROWS, f"Singlepart pieces: {len(sp_records):,}")
corrected_row_gdf = gpd.GeoDataFrame(sp_records, crs=TARGET_CRS).reset_index(drop=True)
corrected_row_gdf["ROW_ID"] = np.arange(1, len(corrected_row_gdf) + 1, dtype=np.int64)
log(f"  Final singlepart ROW features: {len(corrected_row_gdf):,}\n  ROW_ID range: 1 - {len(corrected_row_gdf):,}")

# ---- STEP 7 ----
banner("STEP 7: ASSIGN BUFFER SEGMENT IDs TO ROW")
log("  Building buffer-segment spatial index...")
segment_geoms = list(buffer_segments.geometry)
segment_ids = buffer_segments["Segment_ID"].to_numpy()
segment_tree = STRtree(segment_geoms)
log(f"  Buffer segments indexed: {len(segment_geoms):,}")
assigned_ids = []
total, t0 = len(corrected_row_gdf), time.time()
for i, geom in enumerate(corrected_row_gdf.geometry, start=1):
    sid = pd.NA
    if geom is not None and not geom.is_empty:
        rp = geom.representative_point()
        for k in segment_tree.query(geom):
            sg = segment_geoms[int(k)]
            try:
                if sg is not None and sg.covers(rp):
                    sid = int(segment_ids[int(k)]); break
            except Exception:
                continue
    assigned_ids.append(sid)
    progress(i, total, t0, LOG_EVERY_ROWS,
             f"Segment IDs assigned: {sum(1 for s in assigned_ids if s is not pd.NA):,}")
corrected_row_gdf["Segment_ID"] = pd.array(assigned_ids, dtype="Int64")

# ---- STEP 8 ----
banner("STEP 8: PREPARE 100-FT TRANSECTS")
transects_100ft = get_transects(TRANSECTS_100FT_GPKG, "100-ft", HUNDRED_FT_TRANSECT_SPACING_M,
                                HUNDRED_FT_TRANSECT_HALF_LENGTH_M, HUNDRED_FT_TRANSECT_LENGTH_M,
                                HUNDRED_FT_TRANSECT_LENGTH_FT, HUNDRED_FT_TRANSECT_HALF_LENGTH_FT,
                                HUNDRED_FT_TRANSECT_SPACING_FT)
log(f"  Final 100-ft transects: {len(transects_100ft):,}")

# ---- STEP 9 ----
banner("STEP 9: CLIP 100-FT TRANSECTS TO CORRECTED ROW")
clipped_100ft = clip_transects_to_rows(transects_100ft, corrected_row_gdf)
log(f"  Final clipped 100-ft transects: {len(clipped_100ft):,}")

# ---- STEP 10 ----
banner("STEP 10: CALCULATE ROW WIDTH")
if len(clipped_100ft) == 0:
    log("  No clipped transects available.")
    corrected_row_gdf["ROW_Width_Meters"] = np.nan
    corrected_row_gdf["ROW_Width_Feet"] = np.nan
else:
    log("  Calculating transect lengths...")
    clipped_100ft["Width_Meters"] = clipped_100ft.geometry.length
    if WIDTH_IN_FEET: clipped_100ft["Width_Feet"] = clipped_100ft["Width_Meters"] / FT
    log("  Calculating median width for each ROW_ID...")
    width_stats = clipped_100ft.groupby("ROW_ID")["Width_Meters"].median().rename("ROW_Width_Meters")
    corrected_row_gdf = corrected_row_gdf.merge(width_stats, left_on="ROW_ID", right_index=True, how="left")
    if WIDTH_IN_FEET: corrected_row_gdf["ROW_Width_Feet"] = corrected_row_gdf["ROW_Width_Meters"] / FT
    log("  Width calculation complete.")
    log(f"    ROWs with width: {corrected_row_gdf['ROW_Width_Meters'].notna().sum():,}")
    log(f"    ROWs without width: {corrected_row_gdf['ROW_Width_Meters'].isna().sum():,}")

# ---- STEP 11 ----
banner("STEP 11: EXPORT CLIPPED 100-FT TRANSECTS")
log(f"  Output:\n    {CLIPPED_TRANSECTS_100FT_GPKG}")
remove_existing(CLIPPED_TRANSECTS_100FT_GPKG)
clipped_100ft.to_file(CLIPPED_TRANSECTS_100FT_GPKG, driver="GPKG")
log("  Clipped 100-ft transect export complete.")

# ---- STEP 12 ----
banner("STEP 12: EXPORT FINAL ROW WIDTH DATASET")
log(f"  Output:\n    {OUTPUT_GPKG}")
remove_existing(OUTPUT_GPKG)
corrected_row_gdf.to_file(OUTPUT_GPKG, driver="GPKG")
log("  Final ROW-width export complete.")

# ---- FINAL QC ----
log("=" * 70); log("FINAL QC SUMMARY"); log("-" * 70)
log(f"Input ROW features: {len(row_gdf):,}")
log(f"Input interstate centerlines: {len(line_gdf):,}")
log(f"Input buffer features: {len(buffer_gdf):,}")
log(f"1-mile transects: {len(mile_transects):,}")
log(f"Buffer segments: {len(buffer_segments):,}")
log(f"Final singlepart ROW features: {len(corrected_row_gdf):,}")
log(f"100-ft transects: {len(transects_100ft):,}")
log(f"Clipped 100-ft transects: {len(clipped_100ft):,}")
if "ROW_Width_Meters" in corrected_row_gdf.columns:
    vw = corrected_row_gdf["ROW_Width_Meters"].dropna()
    if len(vw) > 0:
        log(f"ROWs with calculated width: {len(vw):,}")
        log(f"Median ROW width: {vw.median() / FT:,.2f} ft")
        log(f"Minimum ROW width: {vw.min() / FT:,.2f} ft")
        log(f"Maximum ROW width: {vw.max() / FT:,.2f} ft")
log("-" * 70)
log("OUTPUTS")
for lbl, p in [("Buffer segments", SEGMENTS_GPKG), ("1-mile transects", MILE_TRANSECTS_GPKG),
               ("100-ft transects", TRANSECTS_100FT_GPKG),
               ("Clipped 100-ft transects", CLIPPED_TRANSECTS_100FT_GPKG),
               ("Final ROW width", OUTPUT_GPKG)]:
    log(f"  {lbl}:\n    {p}")
log("-" * 70)
log(f"Total runtime: {hms(time.time() - script_start)}")
log("=" * 70); log("WORKFLOW COMPLETE"); log("=" * 70)

[2026-10-01 11:53:51] ======================================================================
[2026-10-01 11:53:51] AGCC INTERSTATE ROW WIDTH WORKFLOW
[2026-10-01 11:53:51] ======================================================================
[2026-10-01 11:53:51] Target CRS: EPSG:5070
[2026-10-01 11:53:51] 1-mile transect spacing: 5,280 ft | length: 8,000 ft | half-length: 4,000 ft
[2026-10-01 11:53:51] 100-ft transect spacing: 100 ft | length: 4,000 ft | half-length: 2,000 ft
[2026-10-01 11:53:51] ROW input: C:\Users\KyleSteen.AzureAD\Documents\ROW_Width_workspace\AGCC_Final_ROW_CONUS_10_1.gpkg
Interstate centerlines: C:\Users\KyleSteen.AzureAD\Documents\ROW_Width_workspace\Interstates_CONUS_Final_10_1.gpkg
Interstate buffer: C:\Users\KyleSteen.AzureAD\Documents\ROW_Width_workspace\Interstates_CONUS_Final_10_1_Buffer2000ft.gpkg
[2026-10-01 11:53:51] ======================================================================
[2026-10-01 11:53:51] -------------------------------------------